# Parte 4 — Experimento B: BERTweet
### Workshop: Clasificación de Emociones en Twitter

**Modelo:** `vinai/bertweet-base`  
**Pre-entrenamiento:** 850M tweets en inglés  
**Tokenizador:** BPE entrenado sobre texto de Twitter

BERTweet tiene la misma arquitectura que BERT-base (12 capas, 768 dimensiones, ~110M parámetros), pero fue pre-entrenado desde cero sobre tweets.

> **Nota sobre `normalization=True`:** el tokenizador normaliza las URLs a `HTTPURL` y los @mentions a `@USER` antes de tokenizar, replicando el preprocesamiento usado durante el pre-entrenamiento.

**Prerequisito:** haber ejecutado `part-1-data.ipynb` y `part-2-pipeline.ipynb`

In [ ]:
%run part-2-pipeline.ipynb

## Configuración del experimento

In [ ]:
MODEL_CHECKPOINT = "vinai/bertweet-base"
HF_REPO          = "your-username/tweeteval-emotion-bertweet"  # <-- cambia esto
LR               = 2e-5

### 📝 TODO 4.1 — Tokenizar el dataset con BERTweet

Importante: carga el tokenizador con `normalization=True`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok_bertweet = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, normalization=True)
ds_bertweet  = make_tokenized_dataset(tok_bertweet)
print(ds_bertweet)


### 📝 TODO 4.2 — Cargar el modelo BERTweet

Mismo procedimiento que con DistilBERT pero con `MODEL_CHECKPOINT`. Imprime el número de parámetros y compáralo con DistilBERT.

In [ ]:
model_bertweet = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID
)
total = sum(p.numel() for p in model_bertweet.parameters())
trainable = sum(p.numel() for p in model_bertweet.parameters() if p.requires_grad)
print(f"Parámetros totales: {total/1e6:.1f}M | entrenables: {trainable/1e6:.1f}M")


### 📝 TODO 4.3 — Entrenar y evaluar BERTweet

Repite el mismo proceso del TODO 3.3 con el modelo y dataset de BERTweet.

**Pregunta:** ¿necesitarías cambiar el learning rate para BERTweet vs DistilBERT? ¿Por qué sí o por qué no?

In [ ]:
trainer_bertweet = make_trainer(
    model_bertweet, tok_bertweet, ds_bertweet,
    output_dir="./checkpoints/bertweet", lr=LR,
)
trainer_bertweet.train()
plot_training_curves(trainer_bertweet, title="BERTweet")
metrics_bertweet = full_evaluation(trainer_bertweet, ds_bertweet["test"], model_name="BERTweet")
metrics_bertweet


## Push to Hub

In [ ]:
import os
if os.environ.get("HF_TOKEN"):
    trainer_bertweet.push_to_hub(HF_REPO, commit_message="BERTweet — TweetEval emotion")
    print(f"Modelo en https://huggingface.co/{HF_REPO}")
else:
    trainer_bertweet.save_model("./checkpoints/bertweet/best")
    print("Guardado localmente (sin HF_TOKEN).")
